In [21]:
class SongEvaluator:
    def __init__(self, ground_truths, predictions):
        """
        Initializes the evaluator with the dataset.
        """
        if len(ground_truths) != len(predictions):
            raise ValueError("The number of ground truths and predictions must match.")
            
        self.ground_truths = ground_truths
        self.predictions = predictions
        self.total_queries = len(ground_truths)

    def calculate_top_k(self, k=3):
        """
        Calculates the Top-K Accuracy.
        """
        top_k_hits = 0
        for true_song, pred_list in zip(self.ground_truths, self.predictions):
            if true_song in pred_list[:k]:
                top_k_hits += 1
                
        return top_k_hits / self.total_queries

    def calculate_mrr_at_k(self, k=None):
        """
        Calculates the Mean Reciprocal Rank truncated at K (MRR@K).
        If k is None, evaluates the entire list.
        """
        sum_reciprocal_rank = 0.0
        for true_song, pred_list in zip(self.ground_truths, self.predictions):
            # Truncate the list to top K
            search_list = pred_list[:k] if k is not None else pred_list
            
            if true_song in search_list:
                rank = search_list.index(true_song) + 1  # 1-indexed rank
                sum_reciprocal_rank += 1.0 / rank
                
        return sum_reciprocal_rank / self.total_queries

    def calculate_rank_distribution(self):
        """
        Calculates the percentage distribution of ranks across all provided predictions.
        """
        rank_counts = {}
        misses = 0
        
        for true_song, pred_list in zip(self.ground_truths, self.predictions):
            if true_song in pred_list:
                rank = pred_list.index(true_song) + 1
                rank_counts[rank] = rank_counts.get(rank, 0) + 1
            else:
                misses += 1
                
        distribution = {}
        for rank in sorted(rank_counts.keys()):
            percentage = (rank_counts[rank] / self.total_queries) * 100
            distribution[f"Rank {rank}"] = f"{percentage:.1f}%"
            
        if misses > 0:
            miss_percentage = (misses / self.total_queries) * 100
            distribution["Not Found"] = f"{miss_percentage:.1f}%"
            
        return distribution

    def generate_report_card(self, k_values=[1, 3, 5]):
        """
        Prints a formatted report card evaluating Top-K and MRR@K.
        """
        print(f"Total Queries Evaluated: {self.total_queries}\n")
        
        print("--- Performance at K Thresholds ---")
        # Print a clean table header
        print(f"{'Threshold':<12} | {'Top-K Accuracy':<15} | {'MRR@K'}")
        print("-" * 55)
        
        for k in k_values:
            acc = self.calculate_top_k(k)
            mrr = self.calculate_mrr_at_k(k)
            print(f"K = {k:<8} | {acc:>6.2%} ({acc:.3f})   | {mrr:.3f}")
            
        print("\n--- Overall Rank Distribution ---")
        distribution = self.calculate_rank_distribution()
        for rank, pct in distribution.items():
            print(f"{rank:<12}: {pct}")

In [22]:
# Re-using the 10-query dataset from earlier
true_labels = [
    "Queen - Bohemian Rhapsody", "Michael Jackson - Billie Jean", 
    "The Beatles - Let It Be", "Nirvana - Smells Like Teen Spirit", 
    "Eminem - Lose Yourself", "Adele - Rolling in the Deep", 
    "Daft Punk - Get Lucky", "Fleetwood Mac - Dreams", 
    "Radiohead - Creep", "The Weeknd - Blinding Lights"
]

predicted_labels = [
    ["Queen - Bohemian Rhapsody", "Queen - Don't Stop Me Now", "David Bowie - Space Oddity", "The Beatles - Hey Jude", "Elton John - Rocket Man"],
    ["Prince - Kiss", "Michael Jackson - Billie Jean", "Madonna - Like a Virgin", "Whitney Houston - I Wanna Dance With Somebody", "George Michael - Faith"],
    ["The Beatles - Yesterday", "The Beatles - Hey Jude", "The Beatles - Let It Be", "The Rolling Stones - Angie", "Bob Dylan - Like a Rolling Stone"],
    ["Pearl Jam - Alive", "Soundgarden - Black Hole Sun", "Alice In Chains - Man in the Box", "Stone Temple Pilots - Plush", "Nirvana - Come As You Are"],
    ["Dr. Dre - Still D.R.E.", "50 Cent - In Da Club", "Snoop Dogg - Drop It Like It's Hot", "Eminem - Without Me", "Eminem - Lose Yourself"],
    ["Adele - Someone Like You", "Amy Winehouse - Rehab", "Adele - Hello", "Sam Smith - Stay With Me", "Florence + The Machine - Dog Days Are Over"],
    ["Daft Punk - Get Lucky", "Pharrell Williams - Happy", "The Weeknd - Starboy", "Justice - D.A.N.C.E.", "Kavinsky - Nightcall"],
    ["Eagles - Hotel California", "Fleetwood Mac - The Chain", "Fleetwood Mac - Go Your Own Way", "Fleetwood Mac - Dreams", "Steely Dan - Do It Again"],
    ["Muse - Supermassive Black Hole", "Coldplay - Yellow", "The Killers - Mr. Brightside", "Radiohead - Karma Police", "Blur - Song 2"],
    ["The Weeknd - Blinding Lights", "The Weeknd - Save Your Tears", "Dua Lipa - Don't Start Now", "Post Malone - Circles", "Harry Styles - Watermelon Sugar"]
]

# Initialize the evaluator
evaluator = SongEvaluator(true_labels, predicted_labels)

# Generate the full report
evaluator.generate_report_card(k_values=[1, 3, 5])

Total Queries Evaluated: 10

--- Performance at K Thresholds ---
Threshold    | Top-K Accuracy  | MRR@K
-------------------------------------------------------
K = 1        | 30.00% (0.300)   | 0.300
K = 3        | 50.00% (0.500)   | 0.383
K = 5        | 70.00% (0.700)   | 0.428

--- Overall Rank Distribution ---
Rank 1      : 30.0%
Rank 2      : 10.0%
Rank 3      : 10.0%
Rank 4      : 10.0%
Rank 5      : 10.0%
Not Found   : 30.0%


In [ ]:
# class Evaluator:
#     def evaluate(self, fingerprinter, database, test_chunks, noise_seed) -> list:
#         raise NotImplementedError("Evaluator not yet implemented")